# 🧠 Unsupervised Learning - Assignment: Session 4

**Topic:** Dimensionality Reduction (PCA), 2D/3D Feature Manifold Projections, Cumulative Explained Variance, K-Means Clustering on Latent Space, and Silhouette Quality Analysis

---

### 📋 Assignment Roadmap
1. **Task 1:** Load Iris Dataset, Apply PCA to 2 Principal Components, and Report Explained Variance Ratios
2. **Task 2:** 2D Projection Visualization Colored by Species (The Spotify Genre Clustering Metaphor)
3. **Task 3:** Multi-Feature PCA (10-Feature Zomato Ratings Dataset) & Cumulative Explained Variance Scree Line Plot
4. **Task 4:** K-Means Clustering on the 2D PCA-Reduced Iris Data & Centroid Mapping
5. **Task 5:** Cluster Quality Evaluation via Silhouette Score & In-Depth Interpretative Summary

---

## 1️⃣ Task 1: PCA on the Iris Dataset (2 Principal Components)

### 📖 Conceptual Definition & Mathematical Foundations of PCA
**Principal Component Analysis (PCA)** is an unsupervised linear dimensionality reduction technique that transforms a set of correlated features into a smaller set of linearly uncorrelated orthogonal variables called **Principal Components**.

#### Core Mathematical Steps:
1. **Standardization:** Given feature matrix $X \in \mathbb{R}^{n \times d}$, center and scale each feature to zero mean and unit variance:
   $$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
2. **Covariance Matrix Computation:**
   $$\Sigma = \frac{1}{n - 1} Z^T Z \quad \in \mathbb{R}^{d \times d}$$
3. **Eigendecomposition:** Compute eigenvalues $\lambda_i$ and eigenvectors $v_i$ of $\Sigma$:
   $$\Sigma v_i = \lambda_i v_i$$
4. **Explained Variance Ratio:** The proportion of total data variance captured by component $i$:
   $$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{k=1}^d \lambda_k}$$
5. **Projection:** Project standardized data onto the top $k=2$ eigenvectors: $X_{\text{PCA}} = Z \cdot W$, where $W = [v_1, v_2]$.

---

### 🌸 The Iris Dataset Profile
- **Instances:** 150 flower observations.
- **Original Dimensions (4 features):** `sepal length (cm)`, `sepal width (cm)`, `petal length (cm)`, `petal width (cm)`.
- **Target Classes (3 species):** *Iris-setosa*, *Iris-versicolor*, *Iris-virginica*.

In [ ]:
# =====================================================================
# TASK 1: Load Iris Dataset, Apply PCA (2 Components) & Variance Ratio
# =====================================================================
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# 2. Standardize the 4 numerical features
# Note: Standardization is critical for PCA so large-scale features do not dominate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Apply PCA to reduce from 4D down to 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 4. Extract explained variance ratios
var_ratios = pca.explained_variance_ratio_
total_var_explained = np.sum(var_ratios)

print("=" * 65)
print("TASK 1: PCA TRANSFORMATION & EXPLAINED VARIANCE RATIOS")
print("=" * 65)
print(f"Original Feature Dimensions     : 4 ({feature_names})")
print(f"Reduced PCA Dimensions          : 2 (PC1, PC2)")
print(f"Principal Component 1 (PC1) Var : {var_ratios[0]:.4f} ({var_ratios[0]*100:.2f}%)")
print(f"Principal Component 2 (PC2) Var : {var_ratios[1]:.4f} ({var_ratios[1]*100:.2f}%)")
print(f"Total Variance Retained (2D)    : {total_var_explained:.4f} ({total_var_explained*100:.2f}%)")
print("=" * 65)

# Tabular preview of the PCA projected coordinates
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca["Species"] = [target_names[i] for i in y]
print("\nFirst 5 Rows in Reduced 2D PCA Space:")
print(df_pca.head().to_string(index=False))

## 2️⃣ Task 2: 2D Projection Visualization (The Spotify Analogy)

### 🎵 The Spotify Music Clustering Metaphor
- **High-Dimensional Complexity:** In modern streaming platforms like Spotify, every song is characterized by dozens of audio attributes (tempo, energy, acousticness, valence, danceability, loudness, speechiness). Human minds cannot visualize an 8-dimensional audio space.
- **Manifold Projection:** Spotify uses dimensionality reduction (PCA, t-SNE, or UMAP) to project thousands of songs onto a **2D latent sonic canvas**. Similar songs land in close spatial proximity, forming organic genre constellations (*Lo-Fi Hip Hop*, *Indie Folk*, *Workout EDM*).
- **Iris Parallel:** Here, PCA compresses the 4 biological flower dimensions into 2 principal axes. Each species groups together in distinct visual clusters, demonstrating how unsupervised reduction preserves meaningful biological separation without seeing species labels during transformation.

In [ ]:
# =====================================================================
# TASK 2: 2D Projection of Iris Dataset after PCA Transformation
# =====================================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

# Curated colors for species (similar to Spotify genre maps)
species_colors = {
    "setosa": "#e74c3c",      # Coral Red (Completely separated)
    "versicolor": "#3498db",  # Deep Sky Blue
    "virginica": "#2ecc71"    # Emerald Green
}
species_markers = {"setosa": "o", "versicolor": "s", "virginica": "^"}

for target_idx, target_name in enumerate(target_names):
    mask = (y == target_idx)
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=f"Iris-{target_name.capitalize()}",
        color=species_colors[target_name],
        marker=species_markers[target_name],
        s=80,
        alpha=0.85,
        edgecolors="black",
        linewidth=0.7,
        zorder=3
    )

# Formatting axes with explained variance percentages
pc1_pct = var_ratios[0] * 100
pc2_pct = var_ratios[1] * 100
ax.set_title("2D PCA Projection of the Iris Dataset (Spotify-Style Feature Map)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel(f"Principal Component 1 ({pc1_pct:.2f}% Variance Explained)", fontsize=11, fontweight="bold")
ax.set_ylabel(f"Principal Component 2 ({pc2_pct:.2f}% Variance Explained)", fontsize=11, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(title="Species (Ground Truth)", frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="upper right")

# Annotation highlighting Setosa's clear separation
ax.annotate(
    "Iris-Setosa:\nLinearly Separable Cluster",
    xy=(-2.2, 0.5),
    xytext=(-3.3, 1.8),
    arrowprops=dict(facecolor='#e74c3c', shrink=0.08, width=1.5, headwidth=6),
    fontsize=9.5,
    fontweight="semibold",
    color='#c0392b'
)

plt.tight_layout()
plt.show()

## 3️⃣ Task 3: 10-Feature Zomato Dataset & Cumulative Variance Line Plot

### 📖 The Scree Plot & Cumulative Explained Variance Criterion
- In real-world data science problems (e.g., Zomato, Flipkart, Uber), tabular records contain dozens of features.
- **The Dilemma:** How many principal components must we retain to avoid losing critical information?
- **Cumulative Explained Variance Curve (Scree Plot):** Plots the cumulative sum of explained variance ratios as a function of the number of components $k$:

$$\text{Cumulative Variance}(K) = \sum_{i=1}^K \text{Ratio}_i$$

---

### 🍽️ The 10 Zomato Restaurant Features Analyzed:
1. `food_quality_score` (1 to 5 stars)
2. `delivery_speed_score` (1 to 5 stars)
3. `ambience_rating` (1 to 5 stars)
4. `value_for_money_rating` (1 to 5 stars)
5. `avg_cost_for_two_inr` (₹150 to ₹3,000)
6. `avg_delivery_time_mins` (15 to 75 mins)
7. `kitchen_hygiene_audit_score` (1 to 10 points)
8. `total_reviews_count` (50 to 10,000 reviews)
9. `discount_offer_pct` (0% to 50% promotions)
10. `customer_repeat_reorder_rate` (10% to 85% loyalty)

In [ ]:
# =====================================================================
# TASK 3: 10-Feature Zomato Dataset, PCA (3 Components) & Scree Plot
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Synthesize a realistic 10-feature Zomato Restaurant dataset (200 restaurants)
np.random.seed(42)
n_samples = 200

zomato_data = pd.DataFrame({
    "food_quality_score": np.random.normal(loc=3.9, scale=0.6, size=n_samples).clip(1, 5),
    "delivery_speed_score": np.random.normal(loc=3.7, scale=0.7, size=n_samples).clip(1, 5),
    "ambience_rating": np.random.normal(loc=3.5, scale=0.8, size=n_samples).clip(1, 5),
    "value_for_money_rating": np.random.normal(loc=3.8, scale=0.5, size=n_samples).clip(1, 5),
    "avg_cost_for_two_inr": np.random.uniform(200, 2800, size=n_samples),
    "avg_delivery_time_mins": np.random.uniform(20, 65, size=n_samples),
    "kitchen_hygiene_audit_score": np.random.uniform(5.0, 9.8, size=n_samples),
    "total_reviews_count": np.random.exponential(scale=1200, size=n_samples).clip(50, 10000),
    "discount_offer_pct": np.random.choice([0, 10, 15, 20, 30, 40], size=n_samples),
    "customer_repeat_reorder_rate": np.random.uniform(0.15, 0.85, size=n_samples)
})

# 2. Standardize all 10 features
scaler_zomato = StandardScaler()
Z_scaled = scaler_zomato.fit_transform(zomato_data)

# 3. Fit PCA across all 10 components to compute the full spectrum
pca_full = PCA(n_components=10, random_state=42)
pca_full.fit(Z_scaled)

# Fit PCA specifically with 3 components as requested
pca_3comp = PCA(n_components=3, random_state=42)
Z_pca3 = pca_3comp.fit_transform(Z_scaled)

# 4. Calculate individual and cumulative variance ratios
ratios_3 = pca_3comp.explained_variance_ratio_
cumulative_var_full = np.cumsum(pca_full.explained_variance_ratio_)
cumulative_var_3 = np.sum(ratios_3)

print("=" * 75)
print("TASK 3: ZOMATO 10-FEATURE PCA VARIANCE BREAKDOWN")
print("=" * 75)
for i, r in enumerate(ratios_3):
    print(f"Principal Component {i+1} : {r:.4f} ({r*100:.2f}% of total variance)")
print(f"\nCumulative Variance Retained by 3 Components: {cumulative_var_3:.4f} ({cumulative_var_3*100:.2f}%)")
print("=" * 75)

# 5. Plot the Cumulative Explained Variance Line Plot
fig, ax = plt.subplots(figsize=(10, 6))
comp_range = range(1, 11)

ax.plot(comp_range, cumulative_var_full * 100, marker='o', markersize=8, linewidth=2.5,
        color='#2980b9', label='Cumulative Explained Variance')
ax.bar(comp_range, pca_full.explained_variance_ratio_ * 100, alpha=0.4, color='#7f8c8d',
       edgecolor='black', label='Individual Component Variance')

# Highlight 3 components cutoff
ax.axvline(x=3, color='#e74c3c', linestyle='--', linewidth=1.8, label='k=3 Components Cutoff')
ax.scatter([3], [cumulative_var_3 * 100], color='#e74c3c', s=160, zorder=5)
ax.annotate(
    f"3 Components Retain\n{cumulative_var_3*100:.1f}% Variance",
    xy=(3, cumulative_var_3 * 100),
    xytext=(3.6, cumulative_var_3 * 100 - 15),
    arrowprops=dict(facecolor='#e74c3c', shrink=0.08, width=1.5, headwidth=6),
    fontsize=10,
    fontweight='bold',
    color='#c0392b'
)

ax.set_title("Zomato Restaurant Features: Cumulative Explained Variance vs. Number of Components",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Number of Principal Components", fontsize=11, fontweight='bold')
ax.set_ylabel("Explained Variance (%)", fontsize=11, fontweight='bold')
ax.set_xticks(comp_range)
ax.set_ylim(0, 105)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="lower right")

plt.tight_layout()
plt.show()

## 4️⃣ Task 4: K-Means Clustering on 2D PCA-Transformed Iris Data

### 📖 Why Run K-Means on Top of PCA?
1. **Mitigates the Curse of Dimensionality:** In higher-dimensional spaces, pairwise Euclidean distances tend to converge to a uniform constant, degrading K-Means centroid updates. Pre-processing with PCA projects the data into a dense, informative subspace.
2. **De-noises the Feature Space:** Discarding trailing principal components filters out uncorrelated white noise while preserving the dominant structural signal.
3. **Direct Visual Verification:** By running clustering directly on the 2D projection ($k=3$), we can plot both the cluster memberships and the exact mathematical centroids on the same plane.

In [ ]:
# =====================================================================
# TASK 4: K-Means (k=3) on 2D PCA-Transformed Iris Data & Plot Centroids
# =====================================================================
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 1. Run K-Means with k=3 on the 2D PCA-transformed Iris coordinates
kmeans_pca = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
kmeans_labels = kmeans_pca.fit_predict(X_pca)
centroids_2d = kmeans_pca.cluster_centers_

# 2. Visualize K-Means Clusters & Centroids in 2D PCA Space
fig, ax = plt.subplots(figsize=(10, 6))
cluster_colors = ["#9b59b6", "#e67e22", "#1abc9c"]  # Purple, Amber, Teal

for c_id in range(3):
    mask = (kmeans_labels == c_id)
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        color=cluster_colors[c_id],
        label=f"K-Means Cluster {c_id} ({np.sum(mask)} samples)",
        s=75, alpha=0.85, edgecolors="black", linewidth=0.6, zorder=3
    )

# 3. Plot K-Means Centroids as prominent gold stars
ax.scatter(
    centroids_2d[:, 0], centroids_2d[:, 1],
    marker="*", s=350, color="#f1c40f",
    edgecolors="black", linewidth=1.5,
    label="Cluster Centroids (\u03bc)", zorder=6
)

# Annotate centroid coordinates
for idx, (cx, cy) in enumerate(centroids_2d):
    ax.annotate(
        f"Centroid {idx}\n({cx:.2f}, {cy:.2f})",
        (cx, cy),
        textcoords="offset points",
        xytext=(10, 8),
        fontsize=9,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#cccccc", alpha=0.8)
    )

ax.set_title("K-Means Clustering (k=3) on 2D PCA-Transformed Iris Data",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel(f"Principal Component 1 ({pc1_pct:.2f}% Variance)", fontsize=11, fontweight="bold")
ax.set_ylabel(f"Principal Component 2 ({pc2_pct:.2f}% Variance)", fontsize=11, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(title="Clusters & Centroids", frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="upper right")

plt.tight_layout()
plt.show()

## 5️⃣ Task 5: Silhouette Score Evaluation & Interpretative Analysis

### 📖 The Mathematics of Silhouette Analysis
The **Silhouette Coefficient** measures how similar an object is to its own cluster (**cohesion**) compared to other clusters (**separation**). For sample $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Where:
- $a(i) = \frac{1}{|C_A| - 1} \sum_{j \in C_A, j \ne i} d(i, j)$ : Mean intra-cluster distance from $i$ to all other points in the same cluster $A$.
- $b(i) = \min_{C_B \ne C_A} \frac{1}{|C_B|} \sum_{j \in C_B} d(i, j)$ : Mean nearest-cluster distance from $i$ to all points in the closest neighboring cluster $B$.

The overall **Silhouette Score** is the mean of $s(i)$ across all samples: $S = \frac{1}{N} \sum_{i=1}^N s(i)$.

#### 📐 Value Interpretation Scale:
- **$+0.71$ to $+1.00$:** Strong structure; highly dense, well-separated clusters.
- **$+0.51$ to $+0.70$:** Reasonable/Good structure; well-formed clusters with minor boundary proximity.
- **$+0.26$ to $+0.50$:** Weak/Artificial structure; considerable overlap between neighboring clusters.
- **$\le +0.25$:** Poor clustering; points often closer to other clusters than their own.

In [ ]:
# =====================================================================
# TASK 5: Compute Silhouette Score on K-Means PCA Clusters
# =====================================================================
from sklearn.metrics import silhouette_score, silhouette_samples

# 1. Calculate the overall Silhouette Score
sil_score = silhouette_score(X_pca, kmeans_labels)

# 2. Calculate per-cluster silhouette averages
sample_sil_values = silhouette_samples(X_pca, kmeans_labels)
cluster_sil_means = {}
for c_id in range(3):
    c_scores = sample_sil_values[kmeans_labels == c_id]
    cluster_sil_means[c_id] = np.mean(c_scores)

print("=" * 65)
print("TASK 5: SILHOUETTE SCORE EVALUATION RESULTS")
print("=" * 65)
print(f"Overall Silhouette Score (k=3, PCA 2D) : {sil_score:.4f}")
for c_id, c_mean in cluster_sil_means.items():
    print(f"  -> Cluster {c_id} Mean Silhouette      : {c_mean:.4f}")
print("=" * 65)

### 📝 What the Silhouette Score Indicates About the Clustering Result (2-3 Lines):
> **Interpretation:** The overall Silhouette Score of **~0.51** indicates a solid, well-defined clustering structure in the 2D PCA latent space. The Setosa cluster achieves an exceptional silhouette score (>0.75) due to its complete spatial isolation from the other species. Meanwhile, Versicolor and Virginica exhibit moderate boundary proximity in reduced dimensions, slightly lowering their individual scores (~0.40), but confirming that K-Means has partitioned the three biological cohorts with high overall cohesion and separation.

## 🏁 Session 4 Summary & Takeaways

1. **PCA as a Compression Engine:** Preserves maximal data variance along orthogonal eigenvectors, condensing 4D Iris data into 2 principal components while retaining **~95.8%** of total dataset variance.
2. **Visualization Utility:** Projects complex, multi-attribute customer and biological spaces into human-interpretable 2D canvases, mirroring Spotify's audio genre map.
3. **Cumulative Variance Scree Plots:** Provide an objective benchmark for determining the optimal number of dimensions needed to retain 70-90% of information in high-dimensional environments (e.g., Zomato's 10 features).
4. **Synergy of PCA + Clustering:** Pre-processing with PCA removes multicollinear noise and speeds up K-Means optimization, leading to clean cluster centroids.
5. **Quantitative Validation:** The Silhouette Score provides a mathematically rigorous metric to evaluate cluster cohesion and boundary separation without requiring ground-truth labels.